In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy rustworkx scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Algoritmul cuantic de optimizare aproximativă

*Estimare de utilizare: 22 de minute pe un procesor Heron r3 (NOTĂ: Aceasta este doar o estimare. Durata ta de execuție poate varia.)*
## Context
Acest tutorial demonstrează cum să implementezi **Algoritmul Cuantic de Optimizare Aproximativă (QAOA)** – o metodă iterativă hibridă (cuantică-clasică) – în contextul pattern-urilor Qiskit. Mai întâi vei rezolva problema **Tăieturii Maxime** (sau **Max-Cut**) pentru un graf mic, apoi vei învăța cum să o execuți la scară de utilitate. Toate execuțiile pe hardware din tutorial ar trebui să se încadreze în limita de timp a planului Open, accesibil gratuit.

Problema Max-Cut este o problemă de optimizare dificil de rezolvat (mai precis, este o problemă *NP-hard*), cu diverse aplicații în clustering, știința rețelelor și fizica statistică. Acest tutorial consideră un graf de noduri conectate prin muchii și urmărește să împartă nodurile în două mulțimi astfel încât numărul de muchii traversate de această tăietură să fie maximizat.

![Ilustrarea unei probleme max-cut](../docs/images/tutorials/quantum-approximate-optimization-algorithm/maxcut-illustration.avif)
## Cerințe
Înainte de a începe acest tutorial, asigură-te că ai instalate următoarele:
- Qiskit SDK v1.0 sau mai recent, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 sau mai recent (`pip install qiskit-ibm-runtime`)

În plus, vei avea nevoie de acces la o instanță pe [IBM Quantum Platform](/guides/cloud-setup). Reține că acest tutorial nu poate fi executat pe planul Open, deoarece rulează sarcini de lucru folosind [sesiuni](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session), care sunt disponibile doar cu acces la planul Premium.
## Configurare

In [1]:
import matplotlib.pyplot as plt
import rustworkx as rx
from rustworkx.visualization import mpl_draw as draw_graph
import numpy as np
from scipy.optimize import minimize
from collections import defaultdict
from typing import Sequence


from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import QAOAAnsatz
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import Session, EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

## Partea I. QAOA la scară mică
Prima parte a acestui tutorial folosește o problemă Max-Cut de dimensiuni mici pentru a ilustra pașii necesari rezolvării unei probleme de optimizare cu un calculator cuantic.

Pentru a oferi un context înainte de a mapa problema la un algoritm cuantic, poți înțelege mai bine cum devine problema Max-Cut o problemă de optimizare combinatorie clasică, considerând mai întâi minimizarea unei funcții $f(x)$

$$
\min_{x\in {0, 1}^n}f(x),
$$

unde intrarea $x$ este un vector ale cărui componente corespund fiecărui nod al unui graf. Apoi, constrânge fiecare componentă să fie fie $0$, fie $1$ (reprezentând includerea sau neincluderea în tăietură). Acest exemplu de scară mică folosește un graf cu $n=5$ noduri.

Poți scrie o funcție pentru o pereche de noduri $i,j$ care indică dacă muchia corespunzătoare $(i,j)$ face parte din tăietură. De exemplu, funcția $x_i + x_j - 2 x_i x_j$ este 1 doar dacă unul dintre $x_i$ sau $x_j$ este 1 (ceea ce înseamnă că muchia face parte din tăietură) și zero în caz contrar. Problema maximizării muchiilor din tăietură poate fi formulată ca

$$
\max_{x\in {0, 1}^n} \sum_{(i,j)} x_i + x_j - 2 x_i x_j,
$$

care poate fi rescrisă ca o minimizare de forma

$$
\min_{x\in {0, 1}^n} \sum_{(i,j)}  2 x_i x_j - x_i - x_j.
$$

Minimul lui $f(x)$ în acest caz apare atunci când numărul de muchii traversate de tăietură este maxim. După cum poți vedea, nu există nimic legat de calculul cuantic până acum. Trebuie să reformulezi această problemă în ceva pe care un calculator cuantic îl poate înțelege.
Inițializează problema ta creând un graf cu $n=5$ noduri.

In [2]:
n_small = 5

graph = rx.PyGraph()
graph.add_nodes_from(np.arange(0, n_small, 1))
edge_list = [
    (0, 1, 1.0),
    (0, 2, 1.0),
    (0, 4, 1.0),
    (1, 2, 1.0),
    (2, 3, 1.0),
    (3, 4, 1.0),
]
graph.add_edges_from(edge_list)
draw_graph(graph, node_size=600, with_labels=True)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/6ced6bea-0.avif)

### Pasul 1: Maparea intrărilor clasice la o problemă cuantică
Primul pas al pattern-ului este de a mapa problema clasică (graful) în **circuite** și **operatori** cuantici. Pentru aceasta, există trei pași principali:

1. Utilizarea unei serii de reformulări matematice pentru a reprezenta această problemă folosind notația problemelor de Optimizare Binară Pătratică Fără Constrângeri (QUBO).
2. Rescrierea problemei de optimizare ca un Hamiltonian al cărui stare fundamentală corespunde soluției care minimizează funcția de cost.
3. Crearea unui Circuit cuantic care va pregăti starea fundamentală a acestui Hamiltonian printr-un proces similar cu recoacerea cuantică.

**Notă:** În metodologia QAOA, vrei în cele din urmă să ai un operator (**Hamiltonian**) care reprezintă **funcția de cost** a algoritmului nostru hibrid, precum și un Circuit parametrizat (**Ansatz**) care reprezintă stări cuantice cu soluții candidate la problemă. Poți eșantiona din aceste stări candidate și apoi le poți evalua folosind funcția de cost.

#### Graf &rarr; problemă de optimizare
Primul pas al mapării este o schimbare de notație. Cele ce urmează exprimă problema în notație QUBO:

$$
\min_{x\in {0, 1}^n}x^T Q x,
$$

unde $Q$ este o matrice $n\times n$ de numere reale, $n$ corespunde numărului de noduri din graful tău, $x$ este vectorul variabilelor binare introduse mai sus, iar $x^T$ indică transpusa vectorului $x$.

```
Maximize
 -2*x_0*x_1 - 2*x_0*x_2 - 2*x_0*x_4 - 2*x_1*x_2 - 2*x_2*x_3 - 2*x_3*x_4 + 3*x_0
 + 2*x_1 + 3*x_2 + 2*x_3 + 2*x_4

Subject to
  No constraints

  Binary variables (5)
    x_0 x_1 x_2 x_3 x_4
```
### Problemă de optimizare &rarr; Hamiltonian
Poți apoi reformula problema QUBO ca un **Hamiltonian** (aici, o matrice care reprezintă energia unui sistem):

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

<details>
<summary>
**Pași de reformulare din problema QAOA la Hamiltonian**
</summary>

Pentru a demonstra cum poate fi rescrisă problema QAOA în acest mod, înlocuiește mai întâi variabilele binare $x_i$ cu un nou set de variabile $z_i\in{-1, 1}$ prin

$$
x_i = \frac{1-z_i}{2}.
$$

Aici poți vedea că dacă $x_i$ este $0$, atunci $z_i$ trebuie să fie $1$. Când $x_i$-urile sunt substituite de $z_i$-uri în problema de optimizare ($x^TQx$), se poate obține o formulare echivalentă.

$$
x^TQx=\sum_{ij}Q_{ij}x_ix_j \\ =\frac{1}{4}\sum_{ij}Q_{ij}(1-z_i)(1-z_j) \\=\frac{1}{4}\sum_{ij}Q_{ij}z_iz_j-\frac{1}{4}\sum_{ij}(Q_{ij}+Q_{ji})z_i + \frac{n^2}{4}.
$$

Acum, dacă definim $b_i=-\sum_{j}(Q_{ij}+Q_{ji})$, eliminăm prefactorul și termenul constant $n^2$, ajungem la cele două formulări echivalente ale aceleiași probleme de optimizare.

$$
\min_{x\in{0,1}^n} x^TQx\Longleftrightarrow \min_{z\in{-1,1}^n}z^TQz + b^Tz
$$

Aici, $b$ depinde de $Q$. Reține că pentru a obține $z^TQz + b^Tz$ am eliminat factorul 1/4 și un offset constant de $n^2$, care nu joacă niciun rol în optimizare.

Acum, pentru a obține o formulare cuantică a problemei, promovăm variabilele $z_i$ la o matrice Pauli $Z$, cum ar fi o matrice $2\times 2$ de forma

$$
Z_i = \begin{pmatrix}1 & 0 \\ 0 & -1\end{pmatrix}.
$$

Când substituiești aceste matrice în problema de optimizare de mai sus, obții următorul Hamiltonian

$$
H_C=\sum_{ij}Q_{ij}Z_iZ_j + \sum_i b_iZ_i.
$$

*Amintește-ți și că matricele $Z$ sunt încorporate în spațiul computațional al calculatorului cuantic, adică un spațiu Hilbert de dimensiune $2^n\times 2^n$. Prin urmare, termeni precum $Z_iZ_j$ trebuie înțeleși ca produsul tensorial $Z_i\otimes Z_j$ încorporat în spațiul Hilbert $2^n\times 2^n$. De exemplu, într-o problemă cu cinci variabile de decizie, termenul $Z_1Z_3$ se înțelege ca $I\otimes Z_3\otimes I\otimes Z_1\otimes I$, unde $I$ este matricea identitate $2\times 2$.*
</details>

Acest Hamiltonian se numește **Hamiltonianul funcției de cost**. Are proprietatea că starea sa fundamentală corespunde soluției care **minimizează funcția de cost $f(x)$**.
Prin urmare, pentru a rezolva problema ta de optimizare, trebuie acum să pregătești starea fundamentală a lui $H_C$ (sau o stare cu un grad mare de suprapunere cu aceasta) pe calculatorul cuantic. Eșantionând din această stare, cu probabilitate ridicată, vei obține soluția la $min~f(x)$.
Acum să considerăm Hamiltonianul $H_C$ pentru problema **Max-Cut**. Asociem fiecărui vertex al grafului un qubit în starea $|0\rangle$ sau $|1\rangle$, unde valoarea denotă mulțimea din care face parte vertex-ul. Scopul problemei este de a maximiza numărul de muchii $(v_1, v_2)$ pentru care $v_1 = |0\rangle$ și $v_2 = |1\rangle$, sau invers. Dacă asociem operatorul $Z$ cu fiecare qubit, unde

$$
    Z|0\rangle = |0\rangle \qquad Z|1\rangle = -|1\rangle
$$

atunci o muchie $(v_1, v_2)$ face parte din tăietură dacă valoarea proprie a lui $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = -1$; cu alte cuvinte, qubiții asociați lui $v_1$ și $v_2$ sunt diferiți. Similar, $(v_1, v_2)$ nu face parte din tăietură dacă valoarea proprie a lui $(Z_1|v_1\rangle) \cdot (Z_2|v_2\rangle) = 1$. Reține că nu ne interesează starea exactă a qubitului asociat fiecărui vertex, ci doar dacă qubiții sunt identici sau diferiți pe o muchie. Problema Max-Cut ne cere să găsim o atribuire a qubiților pe vertex-uri astfel încât valoarea proprie a urmăritorului Hamiltonian să fie minimizată
$$
    H_C = \sum_{(i,j) \in e} Q_{ij} \cdot Z_i Z_j.
$$

Cu alte cuvinte, $b_i = 0$ pentru orice $i$ în problema Max-Cut. Valoarea lui $Q_{ij}$ denotă ponderea muchiei. În acest tutorial considerăm un graf neponderat, adică $Q_{ij} = 1.0$ pentru toți $i, j$.

In [3]:
def build_max_cut_paulis(
    graph: rx.PyGraph,
) -> list[tuple[str, list[int], float]]:
    """Convert graph edges to a list of ZZ Pauli terms.

    The returned list is in the sparse format expected by
    ``SparsePauliOp.from_sparse_list``: each element is
    ``(pauli_string, qubit_indices, coefficient)``.
    """
    pauli_list = []
    for edge in list(graph.edge_list()):
        weight = graph.get_edge_data(edge[0], edge[1])
        pauli_list.append(("ZZ", [edge[0], edge[1]], weight))
    return pauli_list


max_cut_paulis = build_max_cut_paulis(graph)
cost_hamiltonian = SparsePauliOp.from_sparse_list(max_cut_paulis, n_small)
print("Cost Function Hamiltonian:", cost_hamiltonian)

Cost Function Hamiltonian: SparsePauliOp(['IIIZZ', 'IIZIZ', 'ZIIIZ', 'IIZZI', 'IZZII', 'ZZIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])


#### Build the QAOA ansatz circuit

Use `QAOAAnsatz` to construct the parametrized QAOA circuit from the cost Hamiltonian. Here we use `reps=2` (two QAOA layers, four parameters: $\beta_0, \beta_1, \gamma_0, \gamma_1$).

In [4]:
circuit = QAOAAnsatz(cost_operator=cost_hamiltonian, reps=2)
circuit.measure_all()

circuit.draw("mpl")

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/7bd8c6d4-f40f-4a11-a440-0b26d9021b53-0.avif" alt="Output of the previous code cell" />

In [5]:
circuit.parameters

ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(β[1]), ParameterVectorElement(γ[0]), ParameterVectorElement(γ[1])])

### Step 2: Optimize problem for quantum hardware execution

Transpile the abstract circuit into hardware-native instructions. This step handles qubit mapping, gate decomposition, routing, and error suppression. See the transpilation [documentation](/docs/guides/transpile) for more information.

In [6]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=127
)
print(backend)

# Create pass manager for transpilation. Level 3 is the most aggressive
# preset: slower to transpile, but produces shorter circuits that are
# more robust to hardware noise.
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

candidate_circuit = pm.run(circuit)
candidate_circuit.draw("mpl", fold=False, idle_wires=False)

<IBMBackend('ibm_pittsburgh')>


<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif" alt="Output of the previous code cell" />

### Step 3: Execute using Qiskit primitives

The QAOA optimization loop runs inside a Qiskit Runtime [session](/docs/guides/execution-modes) to keep the device reserved across iterations. An Estimator evaluates $\langle H_C \rangle$ at each step, and a classical optimizer (COBYLA) updates the parameters until convergence.

![Illustration showing the behavior of Single job, Batch, and Session runtime modes.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Define initial parameters and run the optimization loop:

In [7]:
# QAOA doesn't prescribe principled default angles — any bounded choice
# works as a warm start for problems this small. beta and gamma are
# periodic (beta in [0, pi] and gamma in [0, 2*pi] modulo the underlying
# Pauli-rotation periods), and pi/2 and pi are just midpoints of those
# ranges. For harder problems you would typically warm start from known
# good angles or transfer parameters from smaller instances.
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_beta, initial_gamma, initial_gamma]

In [8]:
def cost_func_estimator(params, ansatz, hamiltonian, estimator):
    # transform the observable defined on virtual qubits to
    # an observable defined on all physical qubits
    isa_hamiltonian = hamiltonian.apply_layout(ansatz.layout)

    pub = (ansatz, isa_hamiltonian, params)
    job = estimator.run([pub])

    results = job.result()[0]
    cost = results.data.evs

    objective_func_vals.append(cost)

    return cost

In [9]:
objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    # If using qiskit-ibm-runtime<0.24.0, change `mode=` to `session=`
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    estimator.options.environment.job_tags = ["TUT_QAOA"]

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit, cost_hamiltonian, estimator),
        method="COBYLA",
        tol=1e-2,
    )
    print(result)

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -2.0402211719947774
       x: [ 3.041e+00  1.212e+00  2.081e+00  4.471e+00]
    nfev: 36
   maxcv: 0.0


![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/3f28a422-805c-4d3d-b5f6-62539e9133bd-1.avif)

### Pasul 3: Execuție folosind primitivele Qiskit
În fluxul de lucru QAOA, parametrii optimi QAOA sunt găsiți într-o buclă iterativă de optimizare, care rulează o serie de evaluări ale circuitului și folosește un optimizator clasic pentru a găsi parametrii optimi $\beta_k$ și $\gamma_k$. Această buclă de execuție este realizată prin următorii pași:

1. Definirea parametrilor inițiali
2. Instanțierea unei noi `Session` care conține bucla de optimizare și primitiva folosită pentru eșantionarea circuitului
3. Odată ce este găsit un set optim de parametri, execuția circuitului o dată în plus pentru a obține o distribuție finală care va fi folosită în pasul de post-procesare.
#### Definirea circuitului cu parametri inițiali
Începem cu parametri aleși arbitrar.

In [10]:
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif" alt="Output of the previous code cell" />

#### Definirea Backend-ului și a primitivei de execuție
Folosește **primitivele Qiskit Runtime** pentru a interacționa cu Backend-urile IBM&reg;. Cele două primitive sunt Sampler și Estimator, iar alegerea primitivei depinde de tipul de măsurătoare pe care vrei să o rulezi pe calculatorul cuantic. Pentru minimizarea lui $H_C$, folosește Estimator, deoarece măsurarea funcției de cost este pur și simplu valoarea așteptată $\langle H_C \rangle$.
#### Rulare
Primitivele oferă o varietate de [moduri de execuție](/guides/execution-modes) pentru a programa sarcini de lucru pe dispozitivele cuantice, iar un flux de lucru QAOA rulează iterativ într-o sesiune.

![Ilustrare care arată comportamentul modurilor de execuție Single job, Batch și Session.](../docs/images/tutorials/quantum-approximate-optimization-algorithm/runtime-modes.avif)

Poți introduce funcția de cost bazată pe Sampler în rutina de minimizare SciPy pentru a găsi parametrii optimi.

In [11]:
optimized_circuit = candidate_circuit.assign_parameters(result.x)
optimized_circuit.draw("mpl", fold=False, idle_wires=False)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/2989e76e-4296-4dd8-b065-2b8fced064cf-0.avif" alt="Output of the previous code cell" />

In [12]:
# If using qiskit-ibm-runtime<0.24.0, change `mode=` to `backend=`
sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

sampler.options.environment.job_tags = ["TUT_QAOA"]

pub = (optimized_circuit,)
job = sampler.run([pub], shots=int(1e4))
counts_int = job.result()[0].data.meas.get_int_counts()
counts_bin = job.result()[0].data.meas.get_counts()
shots = sum(counts_int.values())
final_distribution_int = {key: val / shots for key, val in counts_int.items()}
final_distribution_bin = {key: val / shots for key, val in counts_bin.items()}
print(final_distribution_int)

{18: 0.039, 5: 0.0665, 20: 0.0973, 29: 0.0063, 9: 0.0899, 13: 0.0379, 2: 0.0047, 1: 0.0153, 11: 0.0932, 14: 0.0327, 12: 0.0314, 25: 0.0193, 21: 0.0398, 6: 0.0224, 4: 0.0197, 10: 0.0387, 3: 0.0181, 26: 0.07, 17: 0.0327, 19: 0.0332, 22: 0.0914, 24: 0.007, 0: 0.0033, 8: 0.0066, 30: 0.0158, 28: 0.0169, 27: 0.0222, 16: 0.0073, 7: 0.0057, 23: 0.0062, 15: 0.0054, 31: 0.0041}


### Step 4: Post-process and return result in desired classical format

Extract the most likely bitstring from the sampled distribution. This represents the best cut found by QAOA.

In [13]:
# auxiliary functions to sample most likely bitstring
def to_bitstring(integer, num_bits):
    result = np.binary_repr(integer, width=num_bits)
    return [int(digit) for digit in result]


keys = list(final_distribution_int.keys())
values = list(final_distribution_int.values())
most_likely = keys[np.argmax(np.abs(values))]
most_likely_bitstring = to_bitstring(most_likely, len(graph))
most_likely_bitstring.reverse()

print("Result bitstring:", most_likely_bitstring)

Result bitstring: [0, 0, 1, 0, 1]


In [14]:
plt.rcParams.update({"font.size": 10})
final_bits = final_distribution_bin
values = np.abs(list(final_bits.values()))
top_4_values = sorted(values, reverse=True)[:4]
positions = []
for value in top_4_values:
    positions.append(np.where(values == value)[0])
fig = plt.figure(figsize=(11, 6))
ax = fig.add_subplot(1, 1, 1)
plt.xticks(rotation=45)
plt.title("Result Distribution")
plt.xlabel("Bitstrings (reversed)")
plt.ylabel("Probability")
ax.bar(list(final_bits.keys()), list(final_bits.values()), color="tab:grey")
for p in positions:
    ax.get_children()[int(p[0])].set_color("tab:purple")
plt.show()

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/e14ecc92-0.avif)

Odată ce ai găsit parametrii optimi pentru circuit, poți atribui acești parametri și eșantiona distribuția finală obținută cu parametrii optimizați. Aici trebuie folosită primitiva *Sampler*, deoarece distribuția de probabilitate a măsurătorilor de șiruri de biți corespunde tăieturii optime a grafului.

**Notă:** Aceasta înseamnă pregătirea unei stări cuantice $\psi$ în calculator și apoi măsurarea ei. O măsurătoare va collapsa starea într-o singură stare din baza computațională — de exemplu, `010101110000...` — care corespunde unei soluții candidate $x$ la problema noastră inițială de optimizare ($\max f(x)$ sau $\min f(x)$ în funcție de sarcină).

In [15]:
# auxiliary function to plot graphs
def plot_result(G, x):
    colors = ["tab:grey" if i == 0 else "tab:purple" for i in x]
    pos, _default_axes = rx.spring_layout(G), plt.axes(frameon=True)
    rx.visualization.mpl_draw(
        G, node_color=colors, node_size=100, alpha=0.8, pos=pos
    )


plot_result(graph, most_likely_bitstring)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif" alt="Output of the previous code cell" />

Now, calculate the value of the cut:

In [16]:
def evaluate_sample(x: Sequence[int], graph: rx.PyGraph) -> float:
    assert len(x) == len(
        list(graph.nodes())
    ), "The length of x must coincide with the number of nodes in the graph."
    return sum(
        x[u] * (1 - x[v]) + x[v] * (1 - x[u])
        for u, v in list(graph.edge_list())
    )


cut_value = evaluate_sample(most_likely_bitstring, graph)
print("The value of the cut is:", cut_value)

The value of the cut is: 5


For a graph this small, the true optimum is easy to brute-force, so you can double-check the results by comparing the QAOA result against the exact answer.

In [17]:
# Classical baseline: enumerate all 2**n_small bitstrings and take the best cut.
def brute_force_max_cut(graph: rx.PyGraph) -> tuple[int, list[int]]:
    n = len(list(graph.nodes()))
    best_cut = -1
    best_x: list[int] = []
    for i in range(2**n):
        x = [(i >> k) & 1 for k in range(n)]
        cut = evaluate_sample(x, graph)
        if cut > best_cut:
            best_cut = int(cut)
            best_x = x
    return best_cut, best_x


classical_best, classical_x = brute_force_max_cut(graph)
print(f"Classical optimum (brute force): {classical_best}")
print(f"QAOA cut value:                  {cut_value}")

Classical optimum (brute force): 5
QAOA cut value:                  5


### Pasul 4: Post-procesare și returnarea rezultatului în formatul clasic dorit
Pasul de post-procesare interpretează ieșirea eșantionării pentru a returna o soluție la problema ta originală. În acest caz, ești interesat de șirul de biți cu probabilitatea cea mai mare, deoarece acesta determină tăietura optimă. Simetriile din problemă permit patru soluții posibile, iar procesul de eșantionare va returna una dintre ele cu o probabilitate ușor mai mare, dar poți vedea în distribuția trasată mai jos că patru dintre șirurile de biți sunt distinctiv mai probabile decât restul.

In [ ]:
# Precomputed parity lookup table: _PARITY[b] = +1 if popcount(b) is even, else -1.
# We use this to vectorize expectation-value evaluation across all Pauli terms.
_PARITY = np.array(
    [-1 if bin(i).count("1") % 2 else 1 for i in range(256)],
    dtype=np.complex128,
)


def evaluate_sparse_pauli(state: int, observable: SparsePauliOp) -> complex:
    """Expectation value of a SparsePauliOp on a single computational-basis state.

    For a Z-only observable (which QAOA cost Hamiltonians are, after the
    QUBO-to-Hamiltonian mapping), the eigenvalue of each Pauli term on a
    computational-basis state is simply (-1)**popcount(z_mask AND state),
    i.e., the parity of the bitwise-AND of the term's Z-support and the
    measured bitstring.

    This routine packs the Z-support of every Pauli term into bytes, ANDs
    them against the measured state in a single vectorized op, and looks up
    the parity in _PARITY. For a 100-qubit / ~hundreds-of-terms Hamiltonian
    over 10_000 samples, this is dramatically faster than calling
    SparsePauliOp.expectation_value per sample.
    """
    packed_uint8 = np.packbits(observable.paulis.z, axis=1, bitorder="little")
    state_bytes = np.frombuffer(
        state.to_bytes(packed_uint8.shape[1], "little"), dtype=np.uint8
    )
    reduced = np.bitwise_xor.reduce(packed_uint8 & state_bytes, axis=1)
    return np.sum(observable.coeffs * _PARITY[reduced])


def best_solution(samples, hamiltonian):
    """Return the sampled bitstring (as int) with the lowest Hamiltonian cost."""
    min_cost = float("inf")
    min_sol = None
    for bit_str in samples.keys():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        if fval <= min_cost:
            min_cost = fval
            min_sol = candidate_sol
    return min_sol


def _plot_cdf(objective_values: dict, ax, color):
    x_vals = sorted(objective_values.keys(), reverse=True)
    y_vals = np.cumsum([objective_values[x] for x in x_vals])
    ax.plot(x_vals, y_vals, color=color)


def plot_cdf(dist, ax, title):
    _plot_cdf(dist, ax, "C1")
    ax.vlines(min(list(dist.keys())), 0, 1, "C1", linestyle="--")
    ax.set_title(title)
    ax.set_xlabel("Objective function value")
    ax.set_ylabel("Cumulative distribution function")
    ax.grid(alpha=0.3)


def samples_to_objective_values(samples, hamiltonian):
    """Convert the samples to values of the objective function."""
    objective_values = defaultdict(float)
    for bit_str, prob in samples.items():
        candidate_sol = int(bit_str)
        fval = evaluate_sparse_pauli(candidate_sol, hamiltonian).real
        objective_values[fval] += prob
    return objective_values

**Step 1**: Build the graph, cost Hamiltonian, and ansatz.

In [19]:
# Step 1: build the 100-node graph, cost Hamiltonian, and QAOA ansatz.
n_large = 100
graph_100 = rx.PyGraph()
graph_100.add_nodes_from(np.arange(0, n_large, 1))
elist = []
for edge in backend.coupling_map:
    if edge[0] < n_large and edge[1] < n_large:
        elist.append((edge[0], edge[1], 1.0))
graph_100.add_edges_from(elist)

max_cut_paulis_100 = build_max_cut_paulis(graph_100)
cost_hamiltonian_100 = SparsePauliOp.from_sparse_list(
    max_cut_paulis_100, n_large
)

circuit_100 = QAOAAnsatz(cost_operator=cost_hamiltonian_100, reps=1)
circuit_100.measure_all()

**Step 2**: Transpile for the selected hardware backend.

In [20]:
# Step 2: transpile for hardware.
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
candidate_circuit_100 = pm.run(circuit_100)

![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/650875e9-adbc-43bd-9505-556be2566278-0.avif)

#### Vizualizarea celei mai bune tăieturi
Din șirul de biți optim, poți vizualiza această tăietură pe graful original.

In [21]:
# Step 3: run the QAOA optimization loop on the device, then sample the
# final distribution with the optimized parameters.
initial_gamma = np.pi
initial_beta = np.pi / 2
init_params = [initial_beta, initial_gamma]

objective_func_vals = []  # Global variable
with Session(backend=backend) as session:
    estimator = Estimator(mode=session)
    estimator.options.default_shots = 1000

    # Set simple error suppression/mitigation options
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    estimator.options.environment.job_tags = ["TUT_QAOA"]

    result = minimize(
        cost_func_estimator,
        init_params,
        args=(candidate_circuit_100, cost_hamiltonian_100, estimator),
        method="COBYLA",
    )
    print(result)

# Assign optimal parameters and sample the final distribution.
optimized_circuit_100 = candidate_circuit_100.assign_parameters(result.x)

sampler = Sampler(mode=backend)
sampler.options.default_shots = 10000

# Set simple error suppression/mitigation options
sampler.options.dynamical_decoupling.enable = True
sampler.options.dynamical_decoupling.sequence_type = "XY4"
sampler.options.twirling.enable_gates = True
sampler.options.twirling.num_randomizations = "auto"

# Add a unique tag to the job execution
sampler.options.environment.job_tags = ["TUT_QAOA"]

pub = (optimized_circuit_100,)
job = sampler.run([pub], shots=int(1e4))

counts_int = job.result()[0].data.meas.get_int_counts()
shots = sum(counts_int.values())
final_distribution_100_int = {
    key: val / shots for key, val in counts_int.items()
}

 message: Return from COBYLA because the trust region radius reaches its lower bound.
 success: True
  status: 0
     fun: -17.172689238986344
       x: [ 2.574e+00  4.166e+00]
    nfev: 28
   maxcv: 0.0


![Rezultatul celulei de cod anterioare](../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/33135970-8bc4-4fb2-ab87-08726a432ce4-0.avif)

Și calculează valoarea tăieturii:

In [22]:
# Step 4: find the best-cost sample and evaluate its cut value.
best_sol_100 = best_solution(final_distribution_100_int, cost_hamiltonian_100)
best_sol_bitstring_100 = to_bitstring(int(best_sol_100), len(graph_100))
best_sol_bitstring_100.reverse()

print("Result bitstring:", best_sol_bitstring_100)

cut_value_100 = evaluate_sample(best_sol_bitstring_100, graph_100)
print("The value of the cut is:", cut_value_100)

Result bitstring: [1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0]
The value of the cut is: 156


Check that the cost minimized in the optimization loop has converged, and visualize results.

In [23]:
# Plot convergence
plt.figure(figsize=(12, 6))
plt.plot(objective_func_vals)
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

# Visualize the cut
plot_result(graph_100, best_sol_bitstring_100)

# Plot cumulative distribution function
result_dist = samples_to_objective_values(
    final_distribution_100_int, cost_hamiltonian_100
)
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_cdf(result_dist, ax, backend.name)

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-0.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-1.avif" alt="Output of the previous code cell" />

<Image src="../docs/images/tutorials/quantum-approximate-optimization-algorithm/extracted-outputs/large-scale-viz-2.avif" alt="Output of the previous code cell" />

## Partea a II-a. Extinde la scară mare!
Ai acces la numeroase dispozitive cu peste 100 de qubiți pe IBM Quantum&reg; Platform. Alege unul pentru a rezolva Max-Cut pe un graf ponderat cu 100 de noduri. Aceasta este o problemă de „scară utilă". Pașii pentru construirea fluxului de lucru sunt urmați la fel ca mai sus, dar cu un graf mult mai mare.